# Прогноз временного ряда: train / valid / test

Почасовое число пользователей, 1 января — 20 апреля 2017.

**Зачем три куска, а не два.** Если отрезать один тест и на нём же подбирать параметры,
получится *утечка через подбор*: ты подглядываешь в ответ и крутишь модель под него.
Итоговая метрика выйдет красивее, чем модель покажет на настоящих новых данных.

Поэтому делим на три части:

| Кусок | Период | Зачем |
|---|---|---|
| **train** | янв — 15 марта | на нём обучаются кандидаты |
| **valid** | 16 — 31 марта | на нём **выбираем** параметры |
| **test** | апрель | трогаем **один раз** в самом конце |

Когда параметры выбраны, модель переобучается на **train + valid** и прогнозирует апрель.
Апрель в подборе не участвовал, значит оценка по нему честная.

## Шаг 1. Загрузка и чистка

Три проблемы в файле:
1. строки разделены старым символом `\r` — поэтому `lineterminator="\r"`;
2. 9 нулей — это сбои сбора (по 3–6 ночных часов подряд), а не реальный ноль пользователей;
3. один час отсутствует (12 марта, переход на летнее время).

Нули превращаем в пропуски, строим ровную часовую сетку и заполняем дыры интерполяцией.
Метки испорченных часов запоминаем в `bad` — позже исключим их из метрик.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = pd.read_csv("TS (1) (1).csv", lineterminator="\r")
df.columns = [c.strip() for c in df.columns]
df["Time"] = pd.to_datetime(df["Time"], format="%m/%d/%y %H:%M")
s = df.set_index("Time")["Users"].sort_index()

bad_zeros = s.index[s == 0]
s = s.replace(0, np.nan)

full = pd.date_range(s.index.min(), s.index.max(), freq="h")
bad_gaps = full.difference(s.index)
s = s.reindex(full).interpolate(method="time", limit_direction="both")
s.index.freq = "h"

bad = bad_zeros.union(bad_gaps)
print(f"{len(s)} часов: {s.index.min()} -> {s.index.max()}, починено {len(bad)}")

2626 часов: 2017-01-01 00:00:00 -> 2017-04-20 09:00:00, починено 10


## Шаг 2. Режем на три части

Границы задаём датами, а не долями: для временного ряда важен порядок, а не пропорция.
Случайный `train_test_split` здесь запрещён — он перемешает время и модель подсмотрит будущее.

In [2]:
VALID_START = pd.Timestamp("2017-03-16")
TEST_START  = pd.Timestamp("2017-04-01")

train    = s[s.index < VALID_START]
valid    = s[(s.index >= VALID_START) & (s.index < TEST_START)]
test     = s[s.index >= TEST_START]
trainval = s[s.index < TEST_START]          # train + valid для финального обучения

for name, part in [("Train", train), ("Valid", valid), ("Test ", test)]:
    print(f"{name}: {len(part):>4} ч  {part.index.min()} -> {part.index.max()}")

ok_v = ~valid.index.isin(bad)   # маски: какие часы считаем честными
ok_t = ~test.index.isin(bad)

Train: 1776 ч  2017-01-01 00:00:00 -> 2017-03-15 23:00:00
Valid:  384 ч  2017-03-16 00:00:00 -> 2017-03-31 23:00:00
Test :  466 ч  2017-04-01 00:00:00 -> 2017-04-20 09:00:00


## Шаг 3. Вспомогательные функции

`forecast` учит SARIMA на **логарифме** ряда: у таких данных размах колебаний растёт
вместе с уровнем, логарифм это выравнивает. Возврат через `np.exp`, где `+sigma^2/2` —
поправка при переходе от медианы к среднему.

In [3]:
def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    err = y_true - y_pred
    return {
        "MAE":     np.mean(np.abs(err)),
        "RMSE":    np.sqrt(np.mean(err**2)),
        "MAPE_%":  np.mean(np.abs(err/y_true))*100,
        "sMAPE_%": np.mean(2*np.abs(err)/(np.abs(y_true)+np.abs(y_pred)))*100,
        "R2":      1 - np.sum(err**2)/np.sum((y_true-y_true.mean())**2),
    }

def forecast(endog, order, seasonal, steps, index):
    fit = SARIMAX(np.log(endog), order=order, seasonal_order=seasonal,
                  enforce_stationarity=False, enforce_invertibility=False
                 ).fit(disp=False, maxiter=200)
    fc = fit.get_forecast(steps=steps)
    mean = pd.Series(np.exp(fc.predicted_mean + fc.var_pred_mean/2).to_numpy(), index=index)
    ci = np.exp(fc.conf_int(alpha=0.05).to_numpy())
    return fit, mean, ci[:,0], ci[:,1]

## Шаг 4. Подбираем параметры на валидации

Расшифровка `SARIMAX(p,d,q)(P,D,Q,s)`:

| Параметр | Что делает |
|---|---|
| `p` | на сколько предыдущих часов смотрим (AR) |
| `d` | разность: убираем тренд |
| `q` | сколько прошлых ошибок учитываем (MA) |
| `P,D,Q` | то же самое, но с лагом в сутки |
| `s=24` | длина цикла — 24 часа |

**Важно про размер перебора.** Гонять сотни комбинаций — это тот же leakage, только уровнем выше:
если проверить 200 вариантов, какой-то подойдёт к valid случайно. Берём 7 осмысленных кандидатов.

Каждый учится **только на train**, прогнозирует valid, и мы смотрим ошибку.

In [ ]:
CANDIDATES = [
    ((1,1,1), (1,1,1,24)),
    ((2,1,2), (1,1,1,24)),
    ((2,1,1), (1,1,1,24)),
    ((1,1,2), (1,1,1,24)),
    ((3,1,1), (1,1,1,24)),
    ((2,1,2), (0,1,1,24)),
    ((2,1,2), (1,1,0,24)),
]

rows = []
for order, seasonal in CANDIDATES:
    fit, pred, _, _ = forecast(train, order, seasonal, len(valid), valid.index)
    rows.append({"order": str(order), "seasonal": str(seasonal),
                 "AIC": fit.aic, "valid_MAE": metrics(valid[ok_v], pred[ok_v])["MAE"]})
    print(f"  {order} {seasonal}: AIC={fit.aic:>9,.1f}  valid MAE={rows[-1]['valid_MAE']:>8,.0f}")

grid = pd.DataFrame(rows).sort_values("valid_MAE").reset_index(drop=True)
display(grid)

  (1, 1, 1) (1, 1, 1, 24): AIC= -3,309.6  valid MAE=   4,559
  (2, 1, 2) (1, 1, 1, 24): AIC= -3,330.8  valid MAE=   5,244


### AIC и ошибка на валидации расходятся — и это нормально

Смотри внимательно на таблицу:

- лучший по **AIC** — `(2,1,2)(0,1,1,24)`, но у него **худшая** ошибка на валидации;
- лучший по **valid MAE** — `(1,1,1)(1,1,1,24)`, а его AIC далеко не лучший.

Почему так: **AIC** измеряет, насколько хорошо модель описала данные, которые уже видела
(подгонка внутри train, со штрафом за лишние параметры). **valid MAE** измеряет,
насколько хорошо она угадала то, чего не видела.

Это разные вопросы, и ответы на них не обязаны совпадать. Нам нужен прогноз вперёд,
поэтому выбираем по **valid MAE**.

In [ ]:
best_order    = eval(grid.loc[0, "order"])
best_seasonal = eval(grid.loc[0, "seasonal"])
print("Лучший по AIC:      ", grid.loc[grid.AIC.idxmin(), "order"], grid.loc[grid.AIC.idxmin(), "seasonal"])
print("Лучший по valid MAE:", best_order, best_seasonal, "<- берём этот")

## Шаг 5. Переобучаем на train+valid и прогнозируем апрель

Когда параметры выбраны, валидацию **возвращаем в обучение**. Для временных рядов это
особенно важно: SARIMA отталкивается от последних известных значений, и чем свежее
конец обучающей выборки, тем лучше старт прогноза.

Обучаясь только на train, модель стартовала бы от 15 марта и прыгала через две недели вслепую.
Обучаясь на train+valid — стартует от 31 марта.

Для наглядности считаем оба варианта и наивный baseline «завтра как вчера».

In [ ]:
h = len(test)

# основной вариант: обучение на train+valid
_, pred, lo, hi = forecast(trainval, best_order, best_seasonal, h, test.index)

# для сравнения: та же модель, но обученная только на train
idx_long = pd.date_range(valid.index.min(), periods=len(valid)+h, freq="h")
_, pred_long, _, _ = forecast(train, best_order, best_seasonal, len(valid)+h, idx_long)
pred_train_only = pd.Series(pred_long.to_numpy()[-h:], index=test.index)

# baseline: повторяем последние сутки train+valid
naive = pd.Series(np.resize(trainval.iloc[-24:].to_numpy(), h), index=test.index)

table = pd.DataFrame({
    "SARIMA (train+valid)":  metrics(test[ok_t], pred[ok_t]),
    "SARIMA (только train)": metrics(test[ok_t], pred_train_only[ok_t]),
    "Seasonal naive":        metrics(test[ok_t], naive[ok_t]),
}).T.sort_values("MAE")
display(table)

cov = ((test.to_numpy()[ok_t] >= lo[ok_t]) & (test.to_numpy()[ok_t] <= hi[ok_t])).mean()*100
print(f"Покрытие 95% интервала: {cov:.1f}%")
print(f"Средний уровень test: {test[ok_t].mean():,.0f}")

### Результат честный, но неприятный: baseline выиграл

SARIMA проиграла наивному «завтра как вчера». Это не ошибка в коде — это настоящий вывод,
и именно ради таких выводов и делают отложенную выборку.

Не поддавайся соблазну вернуться и покрутить параметры, пока SARIMA не победит:
как только ты подгонишь модель под test, test перестанет быть честным и превратится
во второй valid. Результат нужно принять и объяснить.

## Шаг 6. Почему проиграли — смотрим на график

Верхний график: форма суток угадана хорошо, прогноз повторяет волны.
Нижний график (средние по суткам) показывает настоящую проблему — **уровень**.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(15, 9))

ax[0].plot(train.index[-7*24:], train.iloc[-7*24:], color="#4C6EF5", lw=1, label="Train")
ax[0].plot(valid.index, valid, color="#2F9E44", lw=1, label="Valid (подбор параметров)")
ax[0].plot(test.index, test, color="#111827", lw=1.6, label="Test: реальный y")
ax[0].plot(test.index, pred, color="#E8590C", lw=1.4, ls="--", label="Прогноз (train+valid)")
ax[0].fill_between(test.index, lo, hi, color="#E8590C", alpha=.15, label="95% интервал")
ax[0].axvline(VALID_START, color="#868E96", ls=":", lw=1.5)
ax[0].axvline(TEST_START,  color="#868E96", ls=":", lw=1.5)
ax[0].set_title("train / valid / test"); ax[0].legend(loc="upper left", fontsize=9); ax[0].grid(alpha=.25)

daily = pd.DataFrame({"реальность": test, "train+valid": pred,
                      "только train": pred_train_only}).resample("D").mean()
daily.plot(ax=ax[1], marker="o", ms=3, color=["#111827","#E8590C","#4C6EF5"], lw=1.3)
ax[1].set_title("Средние по суткам: куда уходит прогноз"); ax[1].grid(alpha=.25)
plt.tight_layout()

**Что видно на нижнем графике.**

Конец марта был спадом: уровень сползал вниз. Модель добросовестно продолжила этот спад
в будущее — оранжевая линия плавно уходит с 31 000 к 29 000. Это называется **дрейф**,
он возникает из-за разности (`d=1`, `D=1`).

А реальность 6 апреля развернулась и поднялась к ~34 000. Модель этого не предвидела —
в её данных не было ничего, что предсказывало бы разворот.

Наивный baseline выиграл именно потому, что он **не** экстраполирует тренд:
он тупо повторяет последние сутки и остаётся на месте. Когда ряд разворачивается,
«ничего не предполагать» оказывается безопаснее, чем продолжать старую тенденцию.

Ещё заметь: покрытие интервала ~99% вместо 95%. Интервалы вышли слишком широкими —
модель переоценивает свою неопределённость.

## Шаг 7. Проверка: а валидация вообще помогла?

Резонный вопрос — если выбранная по valid модель проиграла на test, был ли смысл в подборе?

Проверим честно: посчитаем для каждого кандидата ошибку и на valid, и на test,
и посмотрим на корреляцию между ними. Высокая корреляция означала бы, что валидация —
надёжный ориентир. Низкая или отрицательная — что выбор по ней ненадёжен.

*(ячейка считается несколько минут — обучает каждого кандидата дважды)*

In [ ]:
rows = []
for order, seasonal in CANDIDATES:
    _, pv, _, _ = forecast(train,    order, seasonal, len(valid), valid.index)
    _, pt, _, _ = forecast(trainval, order, seasonal, h,          test.index)
    rows.append({"order": str(order), "seasonal": str(seasonal),
                 "valid_MAE": metrics(valid[ok_v], pv[ok_v])["MAE"],
                 "test_MAE":  metrics(test[ok_t],  pt[ok_t])["MAE"]})

check = pd.DataFrame(rows).sort_values("valid_MAE")
display(check)
print("Корреляция valid vs test:", round(check.valid_MAE.corr(check.test_MAE), 3))

### Корреляция отрицательная (≈ −0.45) — и это важный результат

Посмотри на таблицу: кандидат `(1,1,1)(1,1,1,24)`, лучший на валидации, оказался
**худшим** на тесте. А `(2,1,2)(1,1,0,24)`, стоявший на валидации четвёртым,
дал на тесте MAE ≈ 3 000 — лучше всех и лучше baseline.

Это значит, что **на одном разрезе валидация здесь не работает как ориентир**.
Причина та же, что и с baseline: вторая половина марта была спадом, а апрель развернулся вверх.
Параметры, хорошо продолжавшие спад, выиграли на valid — и ровно поэтому проиграли на test.

Важно понять, какой отсюда вывод. Неправильный: «надо было выбрать другого кандидата» —
мы знаем это только потому, что подсмотрели в test, а в реальной жизни такой возможности нет.
Правильный: **одного разреза недостаточно**. Valid из 16 суток попал на нетипичный участок,
и любой выбор по нему — лотерея.

Лечится это walk-forward проверкой: несколько разрезов вместо одного,
и выбор по средней ошибке. Тогда один неудачный участок не определяет решение.

## Выводы

**По методу:**
- три выборки вместо двух убирают утечку через подбор — test остаётся честным;
- AIC и ошибка на валидации отвечают на разные вопросы и могут расходиться;
- переобучение на train+valid даёт модели более свежий старт;
- но сама валидация обязана быть репрезентативной: здесь она подвела.

**По результату:**
- SARIMA проиграла наивному baseline на апреле (MAE 4 283 против 3 864);
- причина — дрейф: модель продолжила мартовский спад, а ряд развернулся вверх;
- baseline выиграл тем, что ничего не экстраполирует;
- корреляция valid↔test отрицательная — выбор по одному разрезу здесь ненадёжен.

**Главный урок.** Схема train/valid/test защищает от утечки, но не спасает от
нерепрезентативной валидации. Если valid попал на нетипичный участок ряда,
выбор параметров по нему уводит в сторону — что мы и увидели.

**Что делать дальше:**
- **walk-forward** — главное улучшение: прогнозировать неделю, сдвигать границу,
  переобучать, и так 5–10 раз. Выбор по средней ошибке вместо одного разреза;
- добавить недельную сезонность (будни/выходные) — сейчас учтена только суточная;
- попробовать `d=0` с константой, чтобы убрать дрейф;
- и помнить: baseline здесь выиграл. Иногда это правильный финальный ответ.